[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ersilia-os/ub-cedd-projects-workshop/blob/main/projects/yellow/notebooks/yellow_chemical_space.ipynb)

# Exploring the chemical space of ACE inhibitors

**Yellow group · Hypertension**

Before training a model it helps to look at the molecules you have. This notebook turns
the curated ACE dataset into a map, using coordinates from the Ersilia model `eos1klk`,
and asks two questions of it: where do the potent molecules sit, and are the natural
products the group wants to test anywhere near them?

## What you will do

- Load the curated ACE dataset together with the chemical space coordinates from `eos1klk`
- Plot how the measured potency is distributed
- Draw the chemical space and colour it by activity
- Find the most common scaffolds, and mark the ones your group is interested in

## Setup

Run the cell below first. In Colab it downloads the workshop repository (including the data) and installs the packages this project needs. It takes about a minute. **Don't change it.**

In [ ]:
PROJECT = "yellow"
NEEDS_GPU = False
import os, sys, shutil, subprocess
if "google.colab" in sys.modules:
    repo_dir = "/content/ub-cedd-projects-workshop"
    if not os.path.exists(repo_dir):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/ersilia-os/ub-cedd-projects-workshop.git", repo_dir], check=True)
    else:
        subprocess.run(["git", "-C", repo_dir, "pull", "--ff-only"], check=True)
    os.chdir(f"{repo_dir}/projects/{PROJECT}")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.getcwd())
for _cached in [m for m in sys.modules if m == "scripts" or m.startswith("scripts.")]:
    del sys.modules[_cached]  # forget helper modules imported before the pull above
has_gpu = shutil.which("nvidia-smi") is not None and subprocess.run(["nvidia-smi"], capture_output=True).returncode == 0
print(f"Python {sys.version.split()[0]} | GPU: {'yes' if has_gpu else 'no'} | Folder: {os.getcwd()}")
if NEEDS_GPU and not has_gpu:
    print("WARNING: this notebook needs a GPU. Go to Runtime > Change runtime type, choose CPU, and run this cell again.")

## 1. The dataset and its coordinates

Two files go into this notebook.

`data/ace_human_curated.csv` is what came out of the curation notebook: one row per
molecule, with its potency as a **pActivity** (the higher the number, the more potent the
molecule) and an `activity` label, 1 for active and 0 for inactive, using a cutoff of
1 micromolar (pActivity 6).

`data/eos1klk_ace_human.csv` is the output of the Ersilia model
[eos1klk](https://github.com/ersilia-os/eos1klk). The model takes a molecule and returns
eight numbers: a pair of coordinates on each of four different maps. The maps were built
from the Ersilia Reference Library, 1.3 million compounds, so a molecule's position says
where it sits among molecules in general, not only among the ones in this dataset.

> **Note:** this is how that second file was made, and how you make one for
> any other set of molecules. Take the one-column SMILES file the curation notebook
> saved (`ace_human_curated_smiles.csv`), run it through `eos1klk` in Ersilia, and put
> the result in the group's Drive folder **Projects/YellowTeam/Data**, named
> `eos1klk_<what the molecules are>.csv`. It then gets copied into `data/` here, and the
> notebook can read it. Nothing in this notebook needs a GPU or an internet connection
> once the file exists.

Load both files and join them, molecule by molecule, on the SMILES.

In [ ]:
import pandas as pd
import stylia
from scripts import chemspace

THRESHOLD = 6.0  # pActivity 6 is the 1 micromolar cutoff used to label the molecules

space = chemspace.load_space("data/ace_human_curated.csv", "data/eos1klk_ace_human.csv")
print(f"{len(space):,} molecules, each with coordinates on four maps")
space[["smiles", "pactivity", "activity", "evidence"]].head()

These are the eight coordinate columns. Each pair is one map: `pca_x` and
`pca_y` place the molecule on the PCA map, `umap_x` and `umap_y` on the UMAP map, and so
on. The numbers have no units and mean nothing on their own; only the distance between
two molecules does.

In [ ]:
space.filter(regex="_(x|y)$").head()

## 2. How the measurements are spread

A map is only worth reading if you know what is being mapped. Before plotting any
chemistry, look at the numbers themselves: how potent the molecules are, and how many
of them end up on each side of the cutoff.

Set the plotting style once. Every plot in this notebook uses `stylia`, so
they all come out with the same fonts and colours.

In [ ]:
stylia.set_format("slide")
stylia.set_style("ersilia")
nc = stylia.NamedColors()  # nc.yellow, nc.gray, nc.plum, nc.pink, nc.mint ...

print("Plots in this notebook use the Ersilia style, in the yellow group's colour.")

A histogram cuts the pActivity range into bins and counts how many
molecules fall in each one. The dashed line is the cutoff that separates actives from
inactives.

In [ ]:
fig, axs = stylia.create_figure(1, 1)
ax = axs.next()
ax.hist(space["pactivity"].dropna(), bins=50, range=(2.5, 11), color=nc.yellow)
ax.axvline(THRESHOLD, color=nc.pink, linestyle="--")
stylia.label(ax, xlabel="pActivity", ylabel="Molecules",
             title=f"Measured potency (median {space['pactivity'].median():.2f})")

The distribution is wide and has no single peak, which is normal for data
pulled together from many papers: each paper works on its own series of molecules at its
own potency range. Now count how many molecules sit on each side of the cutoff.

In [ ]:
counts = space["activity"].value_counts().rename({0: "inactive", 1: "active"})
fig, axs = stylia.create_figure(1, 1)
ax = axs.next()
ax.bar(counts.index, counts.values, color=[nc.gray, nc.yellow])
stylia.label(ax, xlabel="", ylabel="Molecules",
             title=f"{counts['active'] / counts.sum():.0%} of the molecules are active")

> **Note:** eleven molecules have no pActivity at all. They are the ones a
> paper reported only as "not active", with no number attached, so they count as
> inactive but cannot go in the histogram. `dropna()` in the cell above is what leaves
> them out.

## 3. The map of chemical space

A molecule has thousands of properties, so it cannot be drawn on a page as it is. A
projection squeezes all of them into two numbers, keeping molecules that are alike close
together. `eos1klk` gives four of them, and they disagree on purpose:

- **PCA** is the plainest: it keeps the big distances honest, so far-apart points really
  are different, but everything piles up in the middle.
- **t-SNE** pulls neighbours together and separates groups clearly. The distance between
  two separate clusters means nothing.
- **UMAP** does something similar but keeps a little more of the overall shape.
- **TMAP** lays the molecules out as a tree, which spreads out sparse regions.

None of them has a right answer. Read them together: a group of molecules that stays
together in all four is a real family.

Start with UMAP, one point per molecule.

In [ ]:
fig, axs = stylia.create_figure(1, 1, width=0.5, height=0.5)
ax = axs.next()
chemspace.plot_points(ax, space, "umap", color=nc.yellow)
chemspace.label_space(ax, "umap", f"{len(space):,} curated ACE molecules")

Now the same molecules on all four maps, side by side.

In [ ]:
fig, axs = stylia.create_figure(2, 2)
for projection, letter in zip(chemspace.PROJECTIONS, "ABCD"):
    ax = axs.next()
    chemspace.plot_points(ax, space, projection, color=nc.yellow, alpha=0.4)
    chemspace.label_space(ax, projection, chemspace.PROJECTIONS[projection], abc=letter)

> **Exercise:** pick a map and describe what you see. How many separate
> clumps are there? Does the same number of clumps show up in the other three maps? The
> rest of this notebook uses UMAP, but every cell below works with `"pca"`, `"tsne"` or
> `"tmap"` instead.

## 4. Potency on the map

The map so far only shows chemistry. Colouring it by what was measured is what makes it
useful: if the potent molecules sit in one part of the map, chemistry and potency go
together, and a model has something to learn. If they are scattered everywhere, small
changes matter more than the overall shape of the molecule.

First the two classes, active and inactive, in two colours.

In [ ]:
fig, axs = stylia.create_figure(1, 1, width=0.5, height=0.5)
ax = axs.next()
for value, color, name in [(0, nc.gray, "inactive"), (1, nc.yellow, "active")]:
    chemspace.plot_points(ax, space[space["activity"] == value], "umap", color=color, label=name)
ax.legend()
chemspace.label_space(ax, "umap", "Active and inactive molecules")

Now the potency itself, as a colour that goes from pale to dark. The eleven
molecules without a number are left out.

In [ ]:
measured = space.dropna(subset=["pactivity"])
cm = stylia.FadingColormap("plum")
cm.fit(measured["pactivity"])

fig, axs = stylia.create_figure(1, 1, width=0.5, height=0.5)
ax = axs.next()
chemspace.plot_points(ax, measured, "umap", color=cm.transform(measured["pactivity"]), alpha=0.8)
chemspace.label_space(ax, "umap", "Darker means more potent")

> **Exercise:** look for a region of the map where the dark points are
> packed together, and one where dark and pale points are mixed. Use the cell below to
> read off the molecules in a region you choose: change the four numbers to the corner
> coordinates of the box you are interested in.

A box drawn by hand. `x_range` and `y_range` are the left-right and
bottom-top edges of the region to look inside.

In [ ]:
x_range, y_range = (-0.20, 0.05), (-0.15, 0.05)  # change these

x, y = chemspace.coordinates(space, "umap")
inside = (x > x_range[0]) & (x < x_range[1]) & (y > y_range[0]) & (y < y_range[1])
print(f"{inside.sum()} molecules in the box | "
      f"{space.loc[inside, 'activity'].mean():.0%} active | "
      f"median pActivity {space.loc[inside, 'pactivity'].median():.2f}")
chemspace.draw_molecules(space.loc[inside, "smiles"].head(4),
                         space.loc[inside, "pactivity"].head(4).round(2))

## 5. The scaffolds behind the clusters

A **scaffold** is what is left of a molecule when every side chain is removed: its rings
and the bits that join them. Two molecules with the same scaffold are variations on one
idea, usually from the same paper or the same medicinal chemistry programme. Scaffolds
are the quickest way to find out what the clumps on the map actually are.

Work out the scaffold of every molecule and count the most common ones.

In [ ]:
space["scaffold"] = chemspace.murcko_scaffolds(space["smiles"])
common = space["scaffold"].value_counts()
print(f"{space['scaffold'].nunique():,} different scaffolds | "
      f"{space['scaffold'].isna().sum()} molecules have no rings at all")
common.head(6).rename("molecules")

This is what those six look like, numbered from the most common one down.

In [ ]:
top = common.head(6)
chemspace.draw_molecules(top.index, [f"{i}: {n} molecules" for i, n in enumerate(top.values, 1)],
                         per_row=3)

Now put them on the map. Each scaffold keeps the number it has in the
drawing above, and everything else stays grey.

In [ ]:
palette = stylia.CategoricalPalette("ersilia").get(len(top))

fig, axs = stylia.create_figure(1, 1, width=0.5, height=0.5)
ax = axs.next()
chemspace.plot_backdrop(ax, space, "umap")
for rank, ((scaffold, size), color) in enumerate(zip(top.items(), palette), 1):
    chemspace.plot_points(ax, space[space["scaffold"] == scaffold], "umap",
                          color=color, label=f"{rank} ({size})", alpha=0.9)
ax.legend()
chemspace.label_space(ax, "umap", "The six most common scaffolds")

> **Exercise:** do the molecules that share a scaffold land in the same
> place on the map? Where they do, the map is telling you about whole molecule families.
> Where one scaffold spreads across the map, the side chains are what the projection is
> responding to.

## 6. Where do your molecules of interest sit?

This is the section to fill in for your own project. The plan is to predict the activity
of natural products, such as indoles and xanthones. Before predicting anything, check
whether molecules like those are in the training data at all: a model asked about a kind
of chemistry it has never seen will still return a number, and that number will be
guesswork.

A ring system is written as a **SMARTS** pattern, a short piece of text describing a
partial molecule that can be searched for inside a bigger one. Two are filled in below
as examples. Add your own, then re-run the cells that follow.

Write the ring systems your group cares about here, as `name: SMARTS`.
Delete the examples if they are not the ones you want.

In [ ]:
PATTERNS = {
    "indole": "c1ccc2[nH]ccc2c1",
    "xanthone": "O=c1c2ccccc2oc2ccccc12",
    # "flavone": "O=c1cc(-c2ccccc2)oc2ccccc12",
    # "your scaffold": "...",
}

found = {name: int(chemspace.has_substructure(space["smiles"], smarts).sum())
         for name, smarts in PATTERNS.items()}
pd.Series(found).rename("molecules in the dataset")

Mark them on the map. Molecules matching none of the patterns stay grey,
so the coloured points are the ones to look at.

In [ ]:
space["pattern"] = chemspace.label_substructures(space["smiles"], PATTERNS)
colors = dict(zip(PATTERNS, stylia.CategoricalPalette("ersilia").get(len(PATTERNS))))

fig, axs = stylia.create_figure(1, 1, width=0.5, height=0.5)
ax = axs.next()
chemspace.plot_backdrop(ax, space, "umap")
for name, color in colors.items():
    subset = space[space["pattern"] == name]
    chemspace.plot_points(ax, subset, "umap", color=color, label=f"{name} ({len(subset)})", alpha=0.9)
ax.legend()
chemspace.label_space(ax, "umap", "Ring systems the group is interested in")

And how potent the matching molecules are, compared with everything else.

In [ ]:
summary = space.groupby("pattern").agg(molecules=("smiles", "size"),
                                       median_pactivity=("pactivity", "median"),
                                       percent_active=("activity", "mean"))
summary["percent_active"] = (summary["percent_active"] * 100).round(0)
summary.sort_values("molecules", ascending=False)

> **Exercise:** add the ring systems from your list of natural products to
> `PATTERNS` and run the three cells again. Then answer, in one sentence each: how many
> molecules of that kind has the dataset already seen, are they active, and where do they
> sit relative to the rest of the map? A pattern with almost no matches is a warning, not
> a dead end: it means the model will be predicting outside what it was trained on, and
> those predictions need checking in the laboratory before anything else.

> **Note:** to put your own natural products on this map, write their
> SMILES into a one-column file, run it through `eos1klk`, put the output in Drive next
> to the file this notebook used, and plot the two sets together, the curated molecules
> as the grey backdrop and yours on top.

## Summary

- You joined the curated ACE dataset to the coordinates from the Ersilia model `eos1klk`
  and drew the group's chemical space four different ways.
- The measured potencies spread over eight orders of magnitude, and about two thirds of
  the molecules are active at the 1 micromolar cutoff.
- The dataset is made of a few large molecule families: the six most common scaffolds
  alone cover a sizeable part of it, and they sit in their own regions of the map.
- Marking indoles and xanthones showed how much of that chemistry the dataset already
  contains, which is what decides whether a model trained on it can say anything useful
  about them.

**Next:** train a classifier and a regressor on this dataset, as the purple group did in
`purple_baseline_models.ipynb`, and come back to this map to see which part of the
chemical space the model gets right.